
Interactive review of Gemini OCR bounding box results

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Image as IPImage, display, HTML
import os
from io import BytesIO


In [ ]:
# Setup project paths
script_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()
project_root = script_dir if (script_dir / "data").exists() else script_dir.parent

ocr_output_file = project_root / "data" / "02_raw_batch_mass" / "ocr_output.jsonl"
preprocessed_dir = project_root / "data" / "01_preprocessed"
preprocessed_meta_csv = preprocessed_dir / "all_metadata.csv"

print(f"Project Root: {project_root}")
print(f"OCR Output File: {ocr_output_file}")
print(f"Preprocessed Data Dir: {preprocessed_dir}")
print(f"\nFile exists: {ocr_output_file.exists()}")

In [ ]:
meta_df = pd.read_csv(preprocessed_meta_csv)
print(f"Total metadata records loaded: {len(meta_df)}")
# meta_df['x_offset'] = pd.to_numeric(meta_df['x_offset'], errors='coerce')
# meta_df['y_offset'] = pd.to_numeric(meta_df['y_offset'], errors='coerce')
# print(type(meta_df['x_offset'][0]))
print(meta_df.head())

In [ ]:
# Load OCR output from JSONL
ocr_data = []
with open(ocr_output_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            ocr_data.append(json.loads(line))

df_ocr = pd.DataFrame(ocr_data)

print(f"Total OCR records loaded: {len(df_ocr)}")
print(f"\nDataframe shape: {df_ocr.shape}")
print(f"\nColumn names: {df_ocr.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_ocr.head())

In [ ]:
# Helper function to find original image file
def find_original_image(state, page_num, col_idx):
    """
    Reconstruct the original image path from preprocessed directory.
    Images are stored as: data/01_preprocessed/{state}/{state}_p{page}_r00_c{col}.jpg
    """
    
    # Construct expected filename pattern
    img_path = preprocessed_dir / state /  f"{state}_p{page_num:02d}_r00_c{col_idx:03d}.jpg"
    if img_path.exists():
        return img_path
    
    return None

# Test the function
test_result = find_original_image(df_ocr.iloc[0]['pub'] if len(df_ocr) > 0 else 'Texas', 1, 0)
print(f"Test image search result: {test_result}")
print(f"Test path exists: {test_result.exists() if test_result else False}")

In [ ]:
# functions for interactions
def update_display(change=None):
    """Update display when item index changes"""
    idx = current_idx.value
    row = df_ocr.iloc[idx]
    
    context_display.value = f"""
    <b>Context:</b><br>
    {df_ocr.loc[df_ocr.index[idx] - 4: df_ocr.index[idx] - 1].to_html()}
    <b>{df_ocr.loc[[df_ocr.index[idx]]].to_html()}</b>
    {df_ocr.loc[df_ocr.index[idx] + 1: df_ocr.index[idx] + 4].to_html()}
    """
    
    # Display image
    img_path = find_original_image(row['pub'], row['page'], row['col'])
    # print(row['text'])
    if img_path:
        meta_row = meta_df.loc[
            (meta_df["pub_id"] == row['pub']) & 
            (meta_df["page_num"] == row['page']) & 
            (meta_df["column"] == row['col'])
        ][["x_offset", "y_offset"]]
        x=row['x'] #- meta_row['x_offset'].iloc[0]
        y=row['y'] #- meta_row['y_offset'].iloc[0]

        img = Image.open(img_path)
        draw_obj = ImageDraw.Draw(img)
        # x=int(x / 1000 * img.width)
        # y=int(y / 1000 * img.height)
        # draw_obj.rectangle([(100, 100), (img.width/2, img.height/2)], outline="red", width=6)
        print((x, y), (y + row["width"], y + row["height"]))
        draw_obj.rectangle([(x, y), (x + row["width"], y + row["height"])], outline="red", width=6)
        with BytesIO() as f:
            img.save(f, format='PNG')
            img_bytes = f.getvalue()
            image_widget.value = img_bytes
            image_widget.width=img.width/2
            image_widget.height=img.height/2
            # image_widget = widgets.Image(
            #     value=img_bytes,
            #     format='png',
            #     width=img.width/2,
            #     height=img.height/2
            # )
 
def on_next(b):
    idx = current_idx.value
    if idx < len(df_ocr) - 1:
        current_idx.value = idx + 1

def on_previous(b):
    if current_idx.value > 0:
        current_idx.value = current_idx.value - 1

In [ ]:

# Create review state tracking dictionary
review_state = {
    'current_index': 0,
}

# Create interactive review interface
current_idx = widgets.IntSlider(value=0, min=0, max=len(df_ocr)-1, step=1, description='Item:', width='500px')
context_display = widgets.HTML(value='')

# display image
image_widget = widgets.Image()
   

current_idx.observe(update_display, names='value')

# Buttons for navigation
btn_next = widgets.Button(description='Next', button_style='info')
btn_previous = widgets.Button(description='Previous', button_style='warning')

btn_next.on_click(on_next)
btn_previous.on_click(on_previous)
buttons_box = widgets.HBox([btn_previous, btn_next])
left_box = widgets.VBox([buttons_box, context_display])

# Display the interface
update_display()
display(widgets.HBox([left_box, image_widget]))